# Stage 5: Baseline Establishment (Zero-Shot)

**Project:** Aspect-Level Customer Review Intelligence via Fine-Tuned Small Language Models

Runs the untouched base model (`qwen2.5:1.5b-instruct`, via local Ollama)
over the 300-sentence held-out test set built in
`01_data_prep.ipynb`, using the project's exact §10.1 prompt template and
greedy decoding, per `../docs/Customer_Review_Intelligence_Review0.md` §12.

**This is the first point in the project that actually calls the local
model.** Everything before this notebook only touched files.

Per §12.1: this baseline is run *before* any fine-tuning, so the evaluation
harness (`src/eval_metrics.py`) gets validated against real output before
anything has been trained, and so the "before" number in the eventual
zero-shot-vs-fine-tuned comparison is uncontaminated by knowing what a
fine-tuned model produces.

**Fine-tuning is a separate, later stage — not started here, pending
explicit go-ahead.**


## 1. Setup

In [1]:
import json
import sys
import time
from pathlib import Path

import pandas as pd

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "src"))

import model_config as mcfg
import eval_metrics as em

PROCESSED_DIR = ROOT / "data" / "processed"
RAW_GEN_DIR = ROOT / "outputs" / "raw_generations" / "zero_shot"
RESULTS_DIR = ROOT / "results"
RAW_GEN_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

TEST_PATH = PROCESSED_DIR / "test.jsonl"
print(f"{'OK ' if TEST_PATH.exists() else 'MISSING':8s} {TEST_PATH}")


OK       /home/sreekanth/Downloads/asmta/data/processed/test.jsonl


## 2. Confirm the model is reachable before committing to a 300-call run

In [2]:
if not mcfg.is_reachable():
    raise RuntimeError(
        "Ollama is not reachable at " + mcfg.OLLAMA_HOST +
        " — start it (`ollama serve`) before running this notebook."
    )

local_models = mcfg.list_local_models()
print("Ollama reachable. Locally pulled models:", local_models)

if mcfg.ACTIVE_MODEL not in local_models:
    raise RuntimeError(
        f"ACTIVE_MODEL '{mcfg.ACTIVE_MODEL}' (src/model_config.py) is not pulled locally. "
        f"Run: ollama pull {mcfg.ACTIVE_MODEL}"
    )
print(f"Active model confirmed present: {mcfg.ACTIVE_MODEL}")


Ollama reachable. Locally pulled models: ['qwen2.5:1.5b-instruct']
Active model confirmed present: qwen2.5:1.5b-instruct


## 3. Load the held-out test set

In [3]:
test_records = []
with open(TEST_PATH) as f:
    for line in f:
        test_records.append(json.loads(line))

print(f"Loaded {len(test_records)} test sentences.")
test_records[0]


Loaded 300 test sentences.


{'sentence_id': '2942',
 'text': 'I Love my new computer!',
 'prompt': '### Instruction:\nExtract every product aspect mentioned in the following review sentence\nand the sentiment expressed towards each. Respond only with JSON in the\nschema: {"aspects":[{"aspect":"...","sentiment":"positive|negative|neutral"}]}\nIf no product aspect is mentioned, return {"aspects":[]}.\n\n### Input:\nI Love my new computer!\n\n### Response:\n',
 'target': {'aspects': [], 'overall': 'neutral'},
 'completion': '{"aspects":[],"overall":"neutral"}'}

## 4. Run zero-shot generation over all test sentences (§12.2)

Greedy decoding (`temperature=0`), generation length capped
(`num_predict=200` — these are short review sentences with short JSON
targets; 200 tokens is generous headroom even for a verbose failure mode),
one call per sentence, no retries — a failed call is recorded as a failure,
not silently retried into a rosier number.


In [4]:
GEN_OPTIONS = {"temperature": 0, "num_predict": 200}

raw_generations = []
t_start = time.time()

for i, rec in enumerate(test_records):
    try:
        resp = mcfg.generate(rec["text"], options=GEN_OPTIONS)
        raw_output = resp.get("response", "")
        eval_count = resp.get("eval_count")
        duration_s = resp.get("total_duration", 0) / 1e9
        error = None
    except Exception as e:  # network hiccup, timeout, etc. — record, don't crash the run
        raw_output = ""
        eval_count = None
        duration_s = None
        error = repr(e)

    raw_generations.append({
        "sentence_id": rec["sentence_id"],
        "text": rec["text"],
        "raw_output": raw_output,
        "eval_count": eval_count,
        "duration_s": duration_s,
        "error": error,
    })

    if (i + 1) % 50 == 0 or (i + 1) == len(test_records):
        elapsed = time.time() - t_start
        print(f"  {i+1}/{len(test_records)} done — {elapsed:.0f}s elapsed "
              f"({(i+1)/elapsed:.2f} sentences/sec so far)")

t_total = time.time() - t_start
n_errors = sum(1 for r in raw_generations if r["error"] is not None)
print(f"\nDone. {len(raw_generations)} generations in {t_total:.1f}s "
      f"({len(raw_generations)/t_total:.2f} sentences/sec). "
      f"{n_errors} call(s) failed outright.")


  50/300 done — 17s elapsed (2.95 sentences/sec so far)


  100/300 done — 34s elapsed (2.94 sentences/sec so far)


  150/300 done — 56s elapsed (2.70 sentences/sec so far)


  200/300 done — 71s elapsed (2.82 sentences/sec so far)


  250/300 done — 87s elapsed (2.87 sentences/sec so far)


  300/300 done — 104s elapsed (2.89 sentences/sec so far)

Done. 300 generations in 104.0s (2.89 sentences/sec). 0 call(s) failed outright.


## 5. Save raw generations verbatim, before any parsing (§12.3)

In [5]:
raw_gen_path = RAW_GEN_DIR / "raw_generations.jsonl"
with open(raw_gen_path, "w") as f:
    for r in raw_generations:
        f.write(json.dumps(r) + "\n")
print(f"Wrote {raw_gen_path} ({raw_gen_path.stat().st_size:,} bytes, {len(raw_generations)} records)")


Wrote /home/sreekanth/Downloads/asmta/outputs/raw_generations/zero_shot/raw_generations.jsonl (113,117 bytes, 300 records)


## 6. Qualitative evidence — concrete raw failures (§12.3)

§12.3: *"Quote three or four of these directly in your write-up. A
concrete failure example communicates far more than a percentage."* Scanned
programmatically for a representative example of each named failure mode,
where present in this run.


In [6]:
import re as _re

examples = {}
for r in raw_generations:
    raw = r["raw_output"]
    if not raw:
        continue
    if "code_fence" not in examples and "```" in raw:
        examples["code_fence"] = r
    if "prose_preamble" not in examples and _re.match(r"^\s*(Sure|Here|Okay|I |Based on)", raw, _re.IGNORECASE):
        examples["prose_preamble"] = r
    if "schema_drift" not in examples:
        parsed = em.parse_lenient(raw)
        if parsed is None and em.parse_strict(raw) is None:
            try:
                obj = json.loads(_re.sub(r"^```(?:json)?|```$", "", raw.strip(), flags=_re.MULTILINE).strip())
                if not (isinstance(obj, dict) and "aspects" in obj):
                    examples["schema_drift"] = r
            except Exception:
                pass
    if "unparseable" not in examples and em.parse_lenient(raw) is None and em.parse_strict(raw) is None:
        examples.setdefault("unparseable_other", r)

for label, r in examples.items():
    print(f"--- {label} ---")
    print(f"input:  {r['text']}")
    print(f"output: {r['raw_output']!r}")
    print()

if not examples:
    print("No failure-mode examples found to quote — the raw generations were all clean.")


--- code_fence ---
input:  I Love my new computer!
output: '```json\n{"aspects":[{"aspect":"product","sentiment":"positive"}]}\n```'

--- schema_drift ---
input:  Since I never really got to use this, I can't comment on anything except what went wrong after trying 2 of them.
output: '```json\n[\n    {\n        "aspect": "product performance",\n        "sentiment": "negative"\n    },\n    {\n        "aspect": "product usability",\n        "sentiment": "negative"\n    },\n    {\n        "aspect": "product quality",\n        "sentiment": "negative"\n    }\n]\n```'

--- unparseable_other ---
input:  Since I never really got to use this, I can't comment on anything except what went wrong after trying 2 of them.
output: '```json\n[\n    {\n        "aspect": "product performance",\n        "sentiment": "negative"\n    },\n    {\n        "aspect": "product usability",\n        "sentiment": "negative"\n    },\n    {\n        "aspect": "product quality",\n        "sentiment": "negative"\n    }\n

## 7. Score — strict and lenient (§12.4)

In [7]:
def to_score_record(r, gold_by_id):
    gold = gold_by_id[r["sentence_id"]]
    return {
        "sentence_id": r["sentence_id"],
        "gold_aspects": gold["target"]["aspects"],
        "raw_output": r["raw_output"],
    }

gold_by_id = {rec["sentence_id"]: rec for rec in test_records}
score_input = [to_score_record(r, gold_by_id) for r in raw_generations]

strict_scores = em.evaluate_strict(score_input)
lenient_scores = em.evaluate_lenient(score_input)

print("STRICT:")
for k, v in strict_scores.items():
    print(f"  {k}: {v:.3f}" if isinstance(v, float) else f"  {k}: {v}")

print("\nLENIENT:")
for k, v in lenient_scores.items():
    print(f"  {k}: {v:.3f}" if isinstance(v, float) else f"  {k}: {v}")


STRICT:
  n_records: 300
  M1_json_validity_rate: 0.127
  M2_aspect_precision: 0.244
  M2_aspect_recall: 0.091
  M2_aspect_f1: 0.133
  M3_polarity_accuracy: 0.842
  M4_joint_precision: 0.205
  M4_joint_recall: 0.077
  M4_joint_f1: 0.112
  M5_empty_case_accuracy: 0.914

LENIENT:
  n_records: 300
  M1_json_validity_rate: 0.883
  M2_aspect_precision: 0.228
  M2_aspect_recall: 0.457
  M2_aspect_f1: 0.304
  M3_polarity_accuracy: 0.779
  M4_joint_precision: 0.178
  M4_joint_recall: 0.356
  M4_joint_f1: 0.237
  M5_empty_case_accuracy: 0.148


## 8. Results table (§15.1, §15.2)

In [8]:
results_rows = []
for parser_name, scores in [("strict", strict_scores), ("lenient", lenient_scores)]:
    results_rows.append({
        "condition": "zero_shot",
        "parser": parser_name,
        "n_records": scores["n_records"],
        "M1_json_validity_rate": scores["M1_json_validity_rate"],
        "M2_aspect_f1": scores["M2_aspect_f1"],
        "M3_polarity_accuracy": scores["M3_polarity_accuracy"],
        "M4_joint_f1": scores["M4_joint_f1"],
        "M5_empty_case_accuracy": scores["M5_empty_case_accuracy"],
    })

results_df = pd.DataFrame(results_rows)
results_df


,condition,parser,n_records,M1_json_validity_rate,M2_aspect_f1,M3_polarity_accuracy,M4_joint_f1,M5_empty_case_accuracy
0,zero_shot,strict,300,0.126667,0.132867,0.842105,0.111888,0.913580
1,zero_shot,lenient,300,0.883333,0.304487,0.778947,0.237179,0.148148


In [9]:
results_path = RESULTS_DIR / "results_table.csv"
results_df.to_csv(results_path, index=False)
print(f"Wrote {results_path}")


Wrote /home/sreekanth/Downloads/asmta/results/results_table.csv


## 9. Efficiency (§15.3) — replaces the earlier *guessed* throughput

In [10]:
measured_throughput = len(raw_generations) / t_total
print(f"Measured throughput: {measured_throughput:.2f} sentences/sec on this hardware "
      f"(RTX 3050 4GB, {mcfg.ACTIVE_MODEL}).")

# notebooks/00_feasibility_and_eda.ipynb guessed 5 sentences/sec for the
# application-layer volume estimate (~30,430 earphone sentences). Replacing
# that guess with what was actually measured here.
n_application_sentences = 30_430
est_minutes = n_application_sentences / measured_throughput / 60
print(f"\nRevised estimate for the {n_application_sentences:,}-sentence application-layer run "
      f"(Stage 8): ~{est_minutes:.0f} minutes at this measured throughput "
      f"(previous notebook's guess assumed 5/sec -> {n_application_sentences/5/60:.0f} min).")


Measured throughput: 2.89 sentences/sec on this hardware (RTX 3050 4GB, qwen2.5:1.5b-instruct).

Revised estimate for the 30,430-sentence application-layer run (Stage 8): ~176 minutes at this measured throughput (previous notebook's guess assumed 5/sec -> 101 min).


### Summary

- Zero-shot baseline run against all 300 held-out test sentences, greedy
  decoding, capped generation length. Raw generations saved verbatim to
  `outputs/raw_generations/zero_shot/raw_generations.jsonl` before any
  parsing — the source of truth if these numbers are ever questioned.
- Strict and lenient M1–M5 both computed and saved to
  `results/results_table.csv` (one row each) — this is the "Zero-Shot"
  column for the eventual results table; the "Fine-Tuned" column gets added
  by a later evaluation notebook, once fine-tuning happens.
- Measured (not guessed) inference throughput on this hardware, used to
  revise the application-layer time estimate from notebook 00.

**Fine-tuning (§13) is the next stage in the doc's plan but is explicitly
on hold pending direct instruction — not started here.**
